In [3]:
import os
os.chdir(r"C:\Users\semih\adult-ensemble")

from pathlib import Path
print("Çalıştığım klasör:", Path.cwd())
print([p.name for p in Path(".").iterdir()])

Çalıştığım klasör: C:\Users\semih\adult-ensemble
['adult.data', 'adult.names', 'adult.test', 'adult.zip', 'Index', 'old.adult.names']


In [4]:
for dosya in ["adult.data", "adult.test"]:
    with open(dosya, encoding="utf-8") as f:
        satirlar = f.read().split("\n")
    print(dosya, "→ toplam satır:", len(satirlar))
    print("ilk 3:", *satirlar[:3], sep="\n  ")
    print("son 3:", *satirlar[-3:], sep="\n  ")
    print()

adult.data → toplam satır: 32563
ilk 3:
  39, State-gov, 77516, Bachelors, 13, Never-married, Adm-clerical, Not-in-family, White, Male, 2174, 0, 40, United-States, <=50K
  50, Self-emp-not-inc, 83311, Bachelors, 13, Married-civ-spouse, Exec-managerial, Husband, White, Male, 0, 0, 13, United-States, <=50K
  38, Private, 215646, HS-grad, 9, Divorced, Handlers-cleaners, Not-in-family, White, Male, 0, 0, 40, United-States, <=50K
son 3:
  52, Self-emp-inc, 287927, HS-grad, 9, Married-civ-spouse, Exec-managerial, Wife, White, Female, 15024, 0, 40, United-States, >50K
  
  

adult.test → toplam satır: 16284
ilk 3:
  |1x3 Cross validator
  25, Private, 226802, 11th, 7, Never-married, Machine-op-inspct, Own-child, Black, Male, 0, 0, 40, United-States, <=50K.
  38, Private, 89814, HS-grad, 9, Married-civ-spouse, Farming-fishing, Husband, White, Male, 0, 0, 50, United-States, <=50K.
son 3:
  35, Self-emp-inc, 182148, Bachelors, 13, Married-civ-spouse, Exec-managerial, Husband, White, Male, 0, 0, 

In [5]:
import sqlite3, csv

cols = ["age", "workclass", "fnlwgt", "education", "education_num",
        "marital_status", "occupation", "relationship", "race", "sex",
        "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"]

con = sqlite3.connect("adult.db")
con.execute("DROP TABLE IF EXISTS raw_adult")
con.execute("CREATE TABLE raw_adult (" + ", ".join(c + " TEXT" for c in cols) + ", source TEXT)")

def load(path, source):
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.reader(f))
    good = [r for r in rows if len(r) == len(cols)]
    con.executemany(
        "INSERT INTO raw_adult VALUES (" + ",".join("?" * (len(cols) + 1)) + ")",
        [r + [source] for r in good])
    print(f"{path}: okunan {len(rows)}, yüklenen {len(good)}, atılan {len(rows) - len(good)}")

load("adult.data", "train")
load("adult.test", "test")
con.commit()

print(con.execute("SELECT source, COUNT(*) FROM raw_adult GROUP BY source").fetchall())
for r in con.execute("SELECT workclass, income FROM raw_adult LIMIT 3"):
    print(repr(r[0]), repr(r[1]))

adult.data: okunan 32562, yüklenen 32561, atılan 1
adult.test: okunan 16283, yüklenen 16281, atılan 2
[('test', 16281), ('train', 32561)]
' State-gov' ' <=50K'
' Self-emp-not-inc' ' <=50K'
' Private' ' <=50K'


In [6]:
import pandas as pd

def sql(q):
    return pd.read_sql_query(q, con)

In [7]:
sql("""
SELECT source, income, COUNT(*) AS adet
FROM raw_adult
GROUP BY source, income
""")

,source,income,adet
0,test,<=50K.,12435
1,test,>50K.,3846
2,train,<=50K,24720
3,train,>50K,7841


In [8]:
sql("""
SELECT
  SUM(workclass = ' ?')      AS workclass_eksik,
  SUM(occupation = ' ?')     AS occupation_eksik,
  SUM(native_country = ' ?') AS native_country_eksik
FROM raw_adult
""")

,workclass_eksik,occupation_eksik,native_country_eksik
0,2799,2809,857


In [9]:
con.execute("DROP TABLE IF EXISTS clean_adult")
con.execute("""
CREATE TABLE clean_adult AS
SELECT
  CAST(age AS INTEGER)            AS age,
  NULLIF(TRIM(workclass), '?')    AS workclass,
  CAST(fnlwgt AS INTEGER)         AS fnlwgt,
  TRIM(education)                 AS education,
  CAST(education_num AS INTEGER)  AS education_num,
  TRIM(marital_status)            AS marital_status,
  NULLIF(TRIM(occupation), '?')   AS occupation,
  TRIM(relationship)              AS relationship,
  TRIM(race)                      AS race,
  TRIM(sex)                       AS sex,
  CAST(capital_gain AS INTEGER)   AS capital_gain,
  CAST(capital_loss AS INTEGER)   AS capital_loss,
  CAST(hours_per_week AS INTEGER) AS hours_per_week,
  NULLIF(TRIM(native_country), '?') AS native_country,
  CASE WHEN REPLACE(TRIM(income), '.', '') = '>50K' THEN 1 ELSE 0 END AS income,
  source
FROM raw_adult
""")
con.commit()

In [10]:
display(sql("SELECT source, income, COUNT(*) AS adet FROM clean_adult GROUP BY source, income"))
display(sql("""
SELECT SUM(workclass IS NULL) AS workclass_null,
       SUM(occupation IS NULL) AS occupation_null,
       SUM(native_country IS NULL) AS native_country_null
FROM clean_adult
"""))

,source,income,adet
0,test,0,12435
1,test,1,3846
2,train,0,24720
3,train,1,7841


,workclass_null,occupation_null,native_country_null
0,2799,2809,857


In [11]:
display(sql("SELECT source, COUNT(*) AS toplam FROM clean_adult GROUP BY source"))
display(sql("SELECT source, COUNT(*) AS tekil FROM (SELECT DISTINCT * FROM clean_adult) GROUP BY source"))

,source,toplam
0,test,16281
1,train,32561


,source,tekil
0,test,16276
1,train,32537


In [12]:
con.execute("DROP TABLE IF EXISTS final_adult")
con.execute("""
CREATE TABLE final_adult AS
SELECT DISTINCT * FROM clean_adult WHERE source = 'train'
UNION ALL
SELECT * FROM clean_adult WHERE source = 'test'
""")
con.commit()

display(sql("SELECT source, COUNT(*) AS adet FROM final_adult GROUP BY source"))

,source,adet
0,test,16281
1,train,32537


In [13]:
display(sql("""
SELECT education_num, education,
       COUNT(*)              AS kisi,
       ROUND(AVG(income), 3) AS yuksek_gelir_orani
FROM final_adult
WHERE source = 'train'
GROUP BY education_num, education
ORDER BY education_num
"""))

,education_num,education,kisi,yuksek_gelir_orani
0,1,Preschool,50,0.000
1,2,1st-4th,166,0.036
2,3,5th-6th,332,0.048
3,4,7th-8th,645,0.062
4,5,9th,514,0.053
5,6,10th,933,0.066
6,7,11th,1175,0.051
7,8,12th,433,0.076
8,9,HS-grad,10494,0.160
9,10,Some-college,7282,0.190


In [14]:
display(sql("""
SELECT sex,
       COUNT(*)              AS kisi,
       ROUND(AVG(income), 3) AS yuksek_gelir_orani
FROM final_adult
WHERE source = 'train'
GROUP BY sex
"""))

,sex,kisi,yuksek_gelir_orani
0,Female,10762,0.110
1,Male,21775,0.306
